# Download các thư viện cần thiết

In [1]:
import os
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Tạo hàm để đọc file parquet (đọc các file parquet - data sau khi được processed)

In [2]:
def read_parquet(train_path: str):
    # Lấy tất cả các file parquet trong thư mục
    files = [os.path.join(train_path, f) for f in os.listdir(train_path) if f.endswith('.parquet')]
    
    # Phân loại các file theo loại tên
    user_chunk_files = [file for file in files if 'user_chunk' in file]
        
    # Đọc các file riêng biệt thành DataFrame
    user_chunk_df = pl.concat([pl.read_parquet(file) for file in user_chunk_files]) if user_chunk_files else None
        
    # Trả về một dictionary chứa các DataFrame
    return user_chunk_df

In [3]:
def read_parquet_item(train_path: str):
    # Lấy tất cả các file parquet trong thư mục
    files = [os.path.join(train_path, f) for f in os.listdir(train_path) if f.endswith('.parquet')]
    
    # Phân loại các file theo loại tên
    item_chunk_files = [file for file in files if 'item_chunk' in file]
        
    # Đọc các file riêng biệt thành DataFrame
    item_chunk_df = pl.concat([pl.read_parquet(file) for file in item_chunk_files]) if item_chunk_files else None
        
    # Trả về một dictionary chứa các DataFrame
    return item_chunk_df

In [4]:
def read_parquet_purchase(train_path: str):
    # Lấy tất cả các file parquet trong thư mục
    files = [os.path.join(train_path, f) for f in os.listdir(train_path) if f.endswith('.parquet')]
    
    # Phân loại các file theo loại tên
    purchase_chunk_files = [file for file in files if 'purchase_history_daily_chunk' in file]
        
    # Đọc các file riêng biệt thành DataFrame
    purchase_chunk_df = pl.concat([pl.read_parquet(file) for file in purchase_chunk_files]) if purchase_chunk_files else None
        
    # Trả về một dictionary chứa các DataFrame
    return purchase_chunk_df

# Tạo hàm lưu file parquet sau mỗi task

In [5]:
def split_and_save_parquet(df, num_files, output_dir):
    """
    Tách DataFrame thành nhiều file Parquet và lưu vào thư mục đích.
    
    :param df: DataFrame cần tách
    :param num_files: Số lượng file Parquet muốn tách
    :param output_dir: Thư mục lưu các file Parquet
    """
    # Đảm bảo thư mục tồn tại
    os.makedirs(output_dir, exist_ok=True)
    
    # Tính số dòng mỗi file sẽ có
    num_rows = df.height
    rows_per_file = num_rows // num_files

    # Tách DataFrame thành các phần và lưu mỗi phần vào một file Parquet
    for i in range(num_files):
        start_row = i * rows_per_file
        # Đảm bảo phần cuối cùng sẽ chứa tất cả các dòng còn lại
        end_row = (i + 1) * rows_per_file if i < num_files - 1 else num_rows
        
        # Tách phần DataFrame
        split_df = df[start_row:end_row]
        
        # Lưu phần DataFrame vào file .parquet
        file_path = os.path.join(output_dir, f"sale_pers.purchase_history_daily_chunk_{i}.parquet")
        split_df.write_parquet(file_path)
        print(f"Đã lưu file: {file_path}")

In [6]:
purchase = read_parquet_purchase(".././dataset")
purchase.head()

timestamp,user_id,item_id,event_type,event_value,price,date_key,quantity,customer_id,created_date,updated_date,channel,payment,location,discount,is_deleted
i64,str,str,str,"decimal[38,4]","decimal[38,4]",i32,i32,i32,datetime[μs],datetime[μs],str,str,i32,"decimal[38,4]",bool
1717185910,"""e9dd339daf179ce86716f0ce734e1d…","""6847000000002""","""Purchase""",1.0000,45000.0000,20240531,1,2731206,2024-05-31 20:05:10.663,2024-05-31 20:05:10.663,"""In-Store""","""VietQR""",687,0.0000,false
1717182347,"""76eeb2afca779fba0bd988a8910074…","""1371000000005""","""Purchase""",1.0000,20000.0000,20240531,1,6778401,2024-05-31 19:05:47.590,2024-05-31 19:05:47.590,"""In-Store""","""Tiền mặt""",340,5000.0000,false
1717184923,"""52c68eecb3ca8241d104867d963616…","""6767000000002""","""Purchase""",1.0000,265000.0000,20240531,1,6589280,2024-05-31 19:48:43.330,2024-05-31 19:48:43.330,"""In-Store""","""Tiền mặt""",105,20000.0000,false
1717096117,"""33a6209df085b3340d77160abe4599…","""4373000000001""","""Purchase""",1.0000,125000.0000,20240530,1,1135573,2024-05-30 19:08:37.647,2024-05-30 19:08:37.647,"""In-Store""","""VNPay""",103,0.0000,false
1717186942,"""49ff163495595f68a3839a2a292963…","""3880000000001""","""Purchase""",2.0000,60000.0000,20240531,2,3732946,2024-05-31 20:22:22.583,2024-05-31 20:22:22.623,"""In-Store""","""Tiền mặt""",127,30000.0000,false


In [7]:
item = read_parquet_item(".././dataset")
item.head()

p_id,item_id,price,category_l1_id,category_l1,category_l2_id,category_l2,category_l3_id,category_l3,category_id,category,description,brand,manufacturer,creation_timestamp,is_deleted,created_date,updated_date,sync_status_id,last_sync_date,sync_error_message,image_url,gender_target,age_group,item_type,gp,weight,color,size,origin,volume,material,sale_status,description_new
i32,str,"decimal[38,4]",i32,str,i32,str,i32,str,i32,str,str,str,str,i64,bool,datetime[μs],datetime[μs],i32,datetime[μs],str,str,str,str,str,"decimal[38,4]",f32,str,str,str,str,str,i32,str
17065,"""0502020000004""",99000.0000,1,"""Babycare""",35,"""Bình sữa, phụ kiện""",7050,"""Núm ty""",7058,"""Núm ty Dr Brown""","""Không xác định""","""Dr.Brown's""","""Không xác định""",1333531544,false,2012-04-04 09:25:44.240,2025-08-18 09:59:19.847,2,2025-07-18 17:59:29.898256,null,"""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""",36828.0000,null,"""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""",0,"""Chi tiết sản phẩm …"
72370,"""0010290040150""",69000.0000,3292,"""Thời trang""",3958,"""Cơ cấu hàng cũ""",7007,"""Thời trang bé trai, bé gái cũ""",6987,"""Bộ quần áo bé gái""","""Không xác định""","""Con Cưng""","""Không xác định""",1503046250,false,2017-08-18 08:50:50.713,2025-09-18 16:05:42.360,null,null,null,"""Không xác định""","""Bé Gái""","""Từ 3Y""","""Bộ quần áo""",0.0000,null,"""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""",0,"""Không xác định"""
31154,"""0008010000015""",45000.0000,351,"""Đồ chơi & Sách""",2033,"""0-1Y""",2118,"""Gặm nướu""",2121,"""Gặm nướu khác""","""- Chất liệu: Sản phẩm được làm…","""Thương hiệu khác""","""Không xác định""",1358501584,false,2013-01-18 09:33:04.260,2025-09-27 00:05:36.233,2,2025-07-18 17:59:29.898256,null,"""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""",14490.0000,null,"""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""",0,"""Chi tiết sản phẩm …"
46123,"""0020010000094""",401000.0000,2222,"""Tã""",2272,"""Merries""",2275,"""Merries""",2276,"""Merries_Sơ Sinh""","""﻿﻿Tã dán Merries size S 82 miế…","""Merries Nhật""","""Không xác định""",1400062039,false,2014-05-14 10:07:19.603,2025-09-27 00:05:36.233,2,2025-07-18 17:59:29.898256,null,"""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""",59749.0000,null,"""Không xác định""","""Không xác định""","""Nhật Bản, Nhật Bản""","""Không xác định""","""Giấy, bột giấy, vải không dệt,…",0,"""Không xác định"""
46127,"""0020010000098""",401000.0000,2222,"""Tã""",2272,"""Merries""",2275,"""Merries""",2278,"""Merries_Tã Quần""","""﻿﻿﻿Bỉm tã quần Merries size M …","""Merries Nhật""","""Không xác định""",1400062040,false,2014-05-14 10:07:20.370,2025-09-27 00:05:36.233,2,2025-07-18 17:59:29.898256,null,"""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""",65764.0000,null,"""Không xác định""","""Không xác định""","""Nhật Bản, Nhật Bản""","""Không xác định""","""Giấy, bột giấy, vải không dệt,…",0,"""Không xác định"""


In [8]:
user = read_parquet(".././dataset")
user.head()

customer_id,gender,location,province,membership,timestamp,created_date,updated_date,sync_status_id,last_sync_date,sync_error_message,region,location_name,install_app,install_date,district,user_id,is_deleted
i32,str,i32,str,str,i64,datetime[μs],datetime[μs],i32,datetime[μs],str,str,str,str,i64,str,str,bool
8307982,"""Nữ""",1177,"""Khánh Hòa""","""Standard""",1738520195,2025-02-02 18:16:35.180,2025-07-07 15:33:10.201316,2,2025-07-16 12:10:21.626252,null,"""Duyên hải Nam Trung Bộ""","""KHO - 203 Quốc Lộ 26""","""In-Store""",1738454400,"""Ninh Hòa""","""90e7a2deabb8f1408ae1d2500bac38…",false
8307983,"""Nữ""",800,"""Đà Nẵng""","""Standard""",1738520198,2025-02-02 18:16:38.493,2025-07-07 15:33:10.201316,2,2025-07-16 12:10:21.626252,null,"""Duyên hải Nam Trung Bộ""","""DNA - 1390 Quảng Xương""","""In-Store""",1738454400,"""Hòa Vang""","""bcf5c962299e13948a5f00ec5ec8b9…",false
8307984,"""Nam""",465,"""Long An""","""Standard""",1738520205,2025-02-02 18:16:45.743,2025-09-24 17:11:26.533,2,2025-07-16 12:10:21.626252,null,"""Đồng bằng sông Cửu Long""","""LAN - 16 Tỉnh lộ 825""","""In-Store""",1738454400,"""Đức Hòa""","""f64e5fe636c027a62cfc1e397386aa…",false
8307985,"""Nữ""",716,"""Bình Dương""","""Standard""",1738520223,2025-02-02 18:17:03.383,2025-07-07 15:33:10.201316,2,2025-07-16 12:10:21.626252,null,"""Đông Nam Bộ""","""BDU - Hội Nghĩa""","""In-Store""",1738454400,"""Tân Uyên""","""367d4d3828bc57a12c4ea52a428955…",false
8307986,"""Nữ""",320,"""Đồng Nai""","""Standard""",1738520230,2025-02-02 18:17:10.243,2025-07-07 15:33:10.201316,2,2025-07-16 12:10:21.626252,null,"""Đông Nam Bộ""","""DON - 22 Ấp 114""","""In-Store""",1738454400,"""Định Quán""","""569c3c9facccff404faa5904d147be…",false


# Load các dataframe cần thiết

In [9]:
purchase_df = read_parquet_purchase(".././preprocessed-feature")
purchase_df.head()

item_id,quantity,customer_id,created_date,location,price,log_price,discount_rate,channel,payment_bucket,time_between_purchases,month,seasonal_trend,product_engagement_level,avg_transaction_amount_per_purchase,segment_name,avg_cat_l1_per_purchase,segment_name_right,top10_co_items,top10_co_items_right,birth_date_step1,age_by_step1_months,birth_date_age_group,age_by_age_group_months,adjusted_milk4mom,age_by_milk4mom_months
str,i32,i32,datetime[μs],i32,f64,f64,f64,str,str,duration[μs],i8,str,str,f64,str,f64,str,list[str],list[str],str,f64,str,f64,str,f64
"""6827000000002""",1,1778845,2024-09-21 15:56:53.233,444,45000.0,10.71444,0.1,"""In-Store""","""qr""",361d 22h 18m 10s 853ms,9,"""Autumn""","""High""",612862.920034,"""Trung cấp""",2.222222,"""Mua vừa""","[""6827000000003"", ""6501000000007"", … ""2803000000012""]","[""6827000000003"", ""6501000000007"", … ""2803000000012""]",null,null,null,null,null,null
"""6848000000004""",1,7097314,2024-09-22 14:40:03.067,662,89000.0,11.396403,0.0,"""In-Store""","""card""",181d 2h 31m 1s 303ms,9,"""Autumn""","""High""",435050.15625,"""Bình dân""",1.5625,"""Mua vừa""","[""6848000000003"", ""6848000000002"", … ""4467000000001""]","[""6848000000003"", ""6848000000002"", … ""4467000000001""]","""2024-09-02 15:17:03.910000""",-14.6,"""2024-09-02 15:17:03.910000""",-14.6,"""2024-12-20""",null
"""6847000000002""",1,2229833,2024-09-21 21:45:05.180,178,45000.0,10.71444,0.0,"""In-Store""","""qr""",289d 22h 36m 47s 337ms,9,"""Autumn""","""High""",185612.989332,"""Bình dân""",1.181818,"""Mua ít""","[""6847000000003"", ""6847000000004"", … ""4680000000002""]","[""6847000000003"", ""6847000000004"", … ""4680000000002""]",null,null,null,null,null,null
"""3496000000051""",1,7870294,2024-09-21 20:36:31.073,103,75000.0,11.225257,0.0,"""In-Store""","""cash""",85d 6m 16s 380ms,9,"""Autumn""","""High""",331041.008379,"""Bình dân""",1.625,"""Mua vừa""","[""3496000000048"", ""0175000000007"", … ""6497000000006""]","[""3496000000048"", ""0175000000007"", … ""6497000000006""]",null,null,null,null,null,null
"""0013000000008""",1,7889987,2024-09-21 15:56:07.900,669,132300.0,11.792835,0.3,"""Web""","""cash""",0µs,9,"""Autumn""","""High""",579615.405405,"""Trung cấp""",3.0,"""Mua nhiều""","[""0175000000007"", ""0203000000004"", … ""0199000000002""]","[""0175000000007"", ""0203000000004"", … ""0199000000002""]",null,null,null,null,null,null


In [10]:
purchase_df = purchase_df.drop('top10_co_items_right')

In [11]:
item_df = read_parquet_item(".././preprocessed-feature")
item_df.head()

item_id,price,category_l1,category,brand_final,target_user_group_final,item_type_final,color_final,origin_final,material_final,sale_status,description_merge,age_bucket_final,price_segment
str,"decimal[38,4]",str,str,str,str,str,str,str,str,i32,str,str,str
"""0502020000004""",99000.0000,"""Babycare""","""Núm ty Dr Brown""","""Dr.Brown's""","""Sơ sinh""",null,"""Đen""","""Ý""","""Silicone""",0,"""Chi tiết sản phẩm …","""1-3M""","""Mid"""
"""0010290040150""",69000.0000,"""Thời trang""","""Bộ quần áo bé gái""","""Con Cưng""","""Bé Gái""","""Bộ quần áo""",null,null,null,0,null,"""2-4Y""","""Mid"""
"""0008010000015""",45000.0000,"""Đồ chơi & Sách""","""Gặm nướu khác""","""Thương hiệu khác""","""Bé Trai""",null,"""Hồng""","""Đức""","""Silicone""",0,"""Chi tiết sản phẩm …",null,"""Low"""
"""0020010000094""",401000.0000,"""Tã""","""Merries_Sơ Sinh""","""Merries Nhật""","""Sơ sinh""",null,"""Đen""","""Nhật Bản, Nhật Bản""","""Giấy, bột giấy, vải không dệt,…",0,"""﻿﻿Tã dán Merries size S 82 miế…","""3-6M""","""High"""
"""0020010000098""",401000.0000,"""Tã""","""Merries_Tã Quần""","""Merries Nhật""","""Sơ sinh""",null,"""Đen""","""Nhật Bản, Nhật Bản""","""Giấy, bột giấy, vải không dệt,…",0,"""﻿﻿﻿Bỉm tã quần Merries size M …","""6-9M""","""High"""


In [12]:
user_df = read_parquet(".././preprocessed-feature")
user_df.head()

customer_id,gender,location,province,membership,region,location_name,install_app,district,milk_segment_preference,diaper_segment_preference
i32,str,i32,str,str,str,str,str,str,i64,i64
8220125,"""Nữ""",240,"""Tiền Giang""","""Gold""","""Đồng bằng sông Cửu Long""","""TGI - 364-365 Nguyễn Huệ""","""In-Store""","""Gò Công""",null,null
8220124,"""Nữ""",996,"""Đồng Nai""","""Standard""","""Đông Nam Bộ""","""DON - 569 Quốc lộ 20""","""In-Store""","""Tân Phú""",null,null
8220138,"""Nữ""",606,"""Hồ Chí Minh""","""Standard""","""Đông Nam Bộ""","""HCM - 385 Bùi Đình Túy""","""In-Store""","""Bình Thạnh""",null,null
8220127,"""Nữ""",746,"""Hà Nội""","""Standard""","""Đồng bằng sông Hồng""","""HNI - 16B-4 Nguyễn Văn Lộc""","""In-Store""","""Hà Đông""",null,null
8220130,"""Nam""",418,"""Hồ Chí Minh""","""Standard""","""Đông Nam Bộ""","""HCM - 1069 Tỉnh Lộ 43""","""In-Store""","""Thủ Đức""",null,null


# XÂY DỰNG BẢNG FEATURE - LABEL

Sắp xếp theo thời gian

In [13]:
purchase_df = purchase_df.sort("created_date")  # mặc định tăng dần theo thời gian
purchase_df.head()

item_id,quantity,customer_id,created_date,location,price,log_price,discount_rate,channel,payment_bucket,time_between_purchases,month,seasonal_trend,product_engagement_level,avg_transaction_amount_per_purchase,segment_name,avg_cat_l1_per_purchase,segment_name_right,top10_co_items,birth_date_step1,age_by_step1_months,birth_date_age_group,age_by_age_group_months,adjusted_milk4mom,age_by_milk4mom_months
str,i32,i32,datetime[μs],i32,f64,f64,f64,str,str,duration[μs],i8,str,str,f64,str,f64,str,list[str],str,f64,str,f64,str,f64
"""2006000000006""",1,4689434,2024-01-01 06:44:59.037,627,35200.0,10.46883,0.451713,"""In-Store""","""cash""",365d 8h 31m 6s 243ms,1,"""Winter""","""High""",1.1825e6,"""Trung cấp""",1.378641,"""Mua ít""","[""4336000000001"", ""2808000000001"", … ""0020020000185""]","""2024-03-01 10:47:18.277000""",-20.766667,"""2024-01-03 19:53:41.550000""",-22.7,"""2025-01-29""",null
"""0020010000440""",1,5279260,2024-01-01 06:48:28.537,547,465000.0,13.049795,0.0,"""In-Store""","""cash""",0µs,1,"""Winter""","""High""",465000.0,"""Trung cấp""",1.0,"""Mua ít""","[""2803000000013"", ""2803000000011"", … ""2808000000001""]",null,null,null,null,null,null
"""2485000000004""",1,4190229,2024-01-01 06:49:32.443,483,469000.0,13.05836,0.0,"""In-Store""","""cash""",364d 8h 53m 19s 84ms,1,"""Winter""","""High""",511970.68006,"""Trung cấp""",1.162791,"""Mua ít""","[""2017000000035"", ""2017000000036"", … ""1512000000004""]",null,null,null,null,null,null
"""2482000000004""",1,6530105,2024-01-01 06:51:11.120,348,525000.0,13.171155,0.0,"""In-Store""","""cash""",360d 7h 56m 39s 227ms,1,"""Winter""","""High""",339501.965496,"""Bình dân""",1.631579,"""Mua vừa""","[""1512000000004"", ""5950000000001"", … ""0203000000004""]",null,null,null,null,null,null
"""6767000000003""",1,6393411,2024-01-01 06:52:48.570,560,265000.0,12.487489,0.070175,"""In-Store""","""cash""",301d 12h 1m 4s 490ms,1,"""Winter""","""High""",386749.068627,"""Bình dân""",1.705882,"""Mua vừa""","[""6767000000002"", ""6768000000003"", … ""5950000000001""]",null,null,null,null,null,null


In [21]:
import polars as pl
from datetime import datetime

def build_feature_label(
    transactions_lf: pl.LazyFrame,
    items_lf: pl.LazyFrame,
    users_lf: pl.LazyFrame,  # hiện tại chưa dùng, nhưng để đúng spec đề bài
    begin_hist: datetime,
    end_hist: datetime,
    begin_recent: datetime,
    end_recent: datetime,
) -> pl.LazyFrame:
    """
    Tạo bảng Feature - Label cho bài toán:
    - X_-1: customer_id
    - X_0: item_id
    - X_1: số lần xuất hiện brand của item_id trong lịch sử (begin_hist - end_hist)
    - X_2: số lần xuất hiện age_group (age_bucket_final) của item_id trong lịch sử
    - X_3: số lần xuất hiện category của item_id trong lịch sử
    - Y: 1 nếu (customer_id, item_id) có giao dịch trong recent (begin_recent - end_recent), ngược lại 0
    """

    # 1. Filter giai đoạn HIST và RECENT trên bảng Transactions
    hist_lf = (
        transactions_lf
        .filter(
            pl.col("created_date").is_between(begin_hist, end_hist, closed="both")
        )
    )

    recent_lf = (
        transactions_lf
        .filter(
            pl.col("created_date").is_between(begin_recent, end_recent, closed="both")
        )
    )

    # 2. Lấy các cột item cần cho feature từ bảng Items
    item_attrs = items_lf.select([
        "item_id",
        "brand_final",
        "age_bucket_final",
        "category",
    ])

    # 3. Gắn thông tin brand / age_group / category vào giao dịch HIST
    hist_enriched = hist_lf.join(item_attrs, on="item_id", how="left")

    # 4. Tính count theo customer_id và (brand / age_group / category)
    #    X_1: số lần brand xuất hiện trong lịch sử
    brand_counts = (
        hist_enriched
        .group_by(["customer_id", "brand_final"])
        .agg(pl.len().alias("brand_counts"))
    )

    #    X_2: số lần age_group (age_bucket_final) xuất hiện trong lịch sử
    age_counts = (
        hist_enriched
        .group_by(["customer_id", "age_bucket_final"])
        .agg(pl.len().alias("age_counts"))
    )

    #    X_3: số lần category xuất hiện trong lịch sử
    category_counts = (
        hist_enriched
        .group_by(["customer_id", "category"])
        .agg(pl.len().alias("category_counts"))
    )

    # 5. Xây tập (customer_id, item_id) candidate
    #    - lấy từ HIST
    hist_pairs = hist_lf.select(["customer_id", "item_id"]).unique()

    #    - lấy từ RECENT (để đảm bảo tất cả cặp có label = 1 đều có mặt)
    recent_pairs = recent_lf.select(["customer_id", "item_id"]).unique()

    #    - union 2 phía
    candidate_pairs = pl.concat([hist_pairs, recent_pairs]).unique()

    # 6. Gắn brand / age_group / category của item vào candidate_pairs
    candidate_enriched = candidate_pairs.join(item_attrs, on="item_id", how="left")

    # 7. Join để tạo X_1, X_2, X_3 cho từng (customer_id, item_id)
    features = (
        candidate_enriched
        # join count theo brand
        .join(
            brand_counts,
            on=["customer_id", "brand_final"],
            how="left",
        )
        # join count theo age_group
        .join(
            age_counts,
            on=["customer_id", "age_bucket_final"],
            how="left",
        )
        # join count theo category
        .join(
            category_counts,
            on=["customer_id", "category"],
            how="left",
        )
        # fill null = 0 (không có lịch sử thì count = 0)
        .with_columns([
            pl.col("brand_counts").fill_null(0),
            pl.col("age_counts").fill_null(0),
            pl.col("category_counts").fill_null(0),
        ])
    )

    # 8. Tạo label Y từ RECENT:
    #    - Nếu (customer_id, item_id) xuất hiện trong khoảng recent → Y = 1
    #    - Ngược lại → Y = 0
    labels = recent_pairs.with_columns(
        pl.lit(1).alias("Y")
    )

    feature_label_lf = (
        features
        .join(labels, on=["customer_id", "item_id"], how="left")
        .with_columns(
            pl.col("Y").fill_null(0).cast(pl.Int8)
        )
        .select([
            pl.col("customer_id"),
            pl.col("item_id"),
            "brand_counts",
            "age_counts",
            "category_counts",
            "Y",
        ])
    )

    return feature_label_lf

In [22]:
from datetime import datetime

begin_hist = datetime(2024, 1, 1)
end_hist = datetime(2024, 10, 31)   # 30/09/2024, không phải 31/09
begin_recent = datetime(2024, 11, 1)
end_recent = datetime(2024, 11, 30)

feature_label_lf = build_feature_label(
    transactions_lf=purchase_df.lazy(),   # convert DataFrame -> LazyFrame
    items_lf=item_df.lazy(),
    users_lf=user_df.lazy(),
    begin_hist=begin_hist,
    end_hist=end_hist,
    begin_recent=begin_recent,
    end_recent=end_recent,
)

feature_label_df = feature_label_lf.collect()
feature_label_df.head()


customer_id,item_id,brand_counts,age_counts,category_counts,Y
i32,str,u32,u32,u32,i8
3818887,"""7167000000002""",2,14,2,0
3498899,"""0180000000011""",16,0,1,0
6062797,"""4697000000002""",6,2,1,0
1936503,"""2483000000004""",4,2,1,0
5939703,"""5420000000003""",5,24,5,0


In [24]:
# Kiểm tra có cặp nào bị trùng hay không
dup_pairs = (
    feature_label_df
    .group_by(["customer_id", "item_id"])
    .agg(pl.count().alias("cnt"))
    .filter(pl.col("cnt") > 1)  # chỉ giữ những cặp xuất hiện > 1 lần
)

dup_pairs

/tmp/ipykernel_2681024/2885721433.py:4: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  .agg(pl.count().alias("cnt"))


customer_id,item_id,cnt
i32,str,u32


In [17]:
feature_label_df['Y'].value_counts()

Y,count
i8,u32
0,19987486
1,2659290


# Xử lý negative sample

Tạo thêm các đặc trưng đếm nhằm phục vụ việc xử lý negative sample có cơ sở hơn

In [86]:
import polars as pl
from datetime import datetime

def build_feature_label(
    transactions_lf: pl.LazyFrame,  # purchase_df.lazy()
    items_lf: pl.LazyFrame,         # item_df.lazy()
    users_lf: pl.LazyFrame,         # chưa dùng, để đúng spec
    begin_hist: datetime,
    end_hist: datetime,
    begin_recent: datetime,
    end_recent: datetime,
) -> pl.LazyFrame:
    """
    Tạo bảng Feature - Label cho bài toán khuyến nghị top-k.

    Output columns:
        - customer_id
        - item_id
        - brand_counts
        - age_counts
        - category_counts
        - segment_counts            (# lần mua cùng (category_l1, segment_name) với item hiện tại)
        - target_user_group_counts
        - Y
    """

    # 0. Làm sạch transactions_lf: bỏ cột segment_name_right nếu tồn tại (do join trước đó sinh ra)
    tx_cols = transactions_lf.columns
    if "segment_name_right" in tx_cols:
        base_tx = transactions_lf.drop("segment_name_right")
    else:
        base_tx = transactions_lf

    # 1. Filter giai đoạn HIST và RECENT trên bảng giao dịch
    hist_lf = (
        base_tx
        .filter(
            pl.col("created_date").is_between(begin_hist, end_hist, closed="both")
        )
    )

    recent_lf = (
        base_tx
        .filter(
            pl.col("created_date").is_between(begin_recent, end_recent, closed="both")
        )
    )

    # 2. Thuộc tính item cơ bản (từ item_df)
    item_attrs = items_lf.select([
        "item_id",
        "brand_final",
        "age_bucket_final",
        "category",
        "category_l1",
        "target_user_group_final",
    ])

    # 3. Mapping item_id -> segment_name đại diện (mode trong HIST)
    item_segment = (
        hist_lf
        .group_by("item_id")
        .agg(pl.col("segment_name").mode().alias("segment_name_list"))
        .with_columns(
            pl.col("segment_name_list").list.first().alias("segment_name")
        )
        .select(["item_id", "segment_name"])
    )

    # 4. HIST đã gắn đầy đủ: brand/age/category/category_l1/segment_name
    hist_enriched = (
        hist_lf
        .join(item_attrs,   on="item_id", how="left")
        .join(item_segment, on="item_id", how="left")
    )

    # 5. Feature 1: brand_counts
    brand_counts = (
        hist_enriched
        .group_by(["customer_id", "brand_final"])
        .agg(pl.len().alias("brand_counts"))
    )

    # 6. Feature 2: age_counts
    age_counts = (
        hist_enriched
        .group_by(["customer_id", "age_bucket_final"])
        .agg(pl.len().alias("age_counts"))
    )

    # 7. Feature 3: category_counts
    category_counts = (
        hist_enriched
        .group_by(["customer_id", "category"])
        .agg(pl.len().alias("category_counts"))
    )

    # 8. Feature: target_user_group_counts
    target_user_group_counts = (
        hist_enriched
        .group_by(["customer_id", "target_user_group_final"])
        .agg(pl.len().alias("target_user_group_counts"))
    )

    # 9. Feature 4 (mới): segment_counts theo (customer, category_l1, segment_name)
    segment_counts = (
        hist_enriched
        .group_by(["customer_id", "category_l1", "segment_name"])
        .agg(pl.len().alias("segment_counts"))
    )

    # 10. Xây tập candidate (customer_id, item_id) từ HIST ∪ RECENT
    hist_pairs = hist_lf.select(["customer_id", "item_id"]).unique()
    recent_pairs = recent_lf.select(["customer_id", "item_id"]).unique()

    candidate_pairs = pl.concat([hist_pairs, recent_pairs]).unique()

    # 11. Enrich candidate với thuộc tính item & segment_name đại diện
    candidate_enriched = (
        candidate_pairs
        .join(item_attrs,   on="item_id", how="left")
        .join(item_segment, on="item_id", how="left")
    )

    # 12. Join tất cả các bảng count để tạo feature set
    features = (
        candidate_enriched
        # brand_counts
        .join(
            brand_counts,
            on=["customer_id", "brand_final"],
            how="left",
        )
        # age_counts
        .join(
            age_counts,
            on=["customer_id", "age_bucket_final"],
            how="left",
        )
        # category_counts
        .join(
            category_counts,
            on=["customer_id", "category"],
            how="left",
        )
        # segment_counts — join theo (customer_id, category_l1, segment_name)
        .join(
            segment_counts,
            on=["customer_id", "category_l1", "segment_name"],
            how="left",
        )
        # target_user_group_counts
        .join(
            target_user_group_counts,
            on=["customer_id", "target_user_group_final"],
            how="left",
        )
        # fill null = 0 cho tất cả count
        .with_columns([
            pl.col("brand_counts").fill_null(0),
            pl.col("age_counts").fill_null(0),
            pl.col("category_counts").fill_null(0),
            pl.col("segment_counts").fill_null(0),
            pl.col("target_user_group_counts").fill_null(0),
        ])
    )

    # 13. Tạo label Y từ RECENT: (customer_id, item_id) có giao dịch trong RECENT -> Y=1
    labels = recent_pairs.with_columns(
        pl.lit(1).alias("Y")
    )

    feature_label_lf = (
        features
        .join(labels, on=["customer_id", "item_id"], how="left")
        .with_columns(
            pl.col("Y").fill_null(0).cast(pl.Int8)
        )
        .select([
            "customer_id",
            "item_id",
            "brand_counts",
            "age_counts",
            "category_counts",
            "segment_counts",
            "target_user_group_counts",
            "Y",
        ])
    )

    return feature_label_lf

In [87]:
begin_hist = datetime(2024, 1, 1)
end_hist = datetime(2024, 10, 31)   # 30/09/2024, không phải 31/09
begin_recent = datetime(2024, 11, 1)
end_recent = datetime(2024, 11, 30)

feature_label_lf = build_feature_label(
    transactions_lf=purchase_df.lazy(),
    items_lf=item_df.lazy(),
    users_lf=user_df.lazy(),
    begin_hist=begin_hist,
    end_hist=end_hist,
    begin_recent=begin_recent,
    end_recent=end_recent,
)
feature_label_df = feature_label_lf.collect()



/tmp/ipykernel_2681024/2431777554.py:28: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  tx_cols = transactions_lf.columns


In [79]:
feature_label_df

customer_id,item_id,brand_counts,age_counts,category_counts,segment_counts,target_user_group_counts,origin_counts,Y
i32,str,u32,u32,u32,u32,u32,u32,i8
3513571,"""1439000000004""",8,35,10,0,41,7,0
5477989,"""3797000000002""",0,0,0,0,0,0,1
6600697,"""2117000000010""",1,2,1,21,10,7,0
7413600,"""0176000000001""",1,0,1,32,7,0,0
7497617,"""0007090000157""",1,6,1,0,3,1,0
…,…,…,…,…,…,…,…,…
5884872,"""1387000000025""",2,12,2,100,4,15,0
4767483,"""2344000000070""",1,3,1,8,2,1,0
4090194,"""6472000000003""",1,2,1,13,2,3,0


In [88]:
feature_label_df['Y'].value_counts()

Y,count
i8,u32
0,19987486
1,2659290
